## **ADA_DC without DC**

In [ ]:
!pip install torch torchvision diffusers transformers accelerate scipy numpy Pillow

# **ADA_DC WITH IFS**

In [ ]:
"""
自适应 DeepCache 实验框架
=========================
三种方法对比：
  1. DDIM Baseline（无缓存，完整推理）
  2. DeepCache（固定缓存间隔）
  3. Adaptive DeepCache（自适应分段 + 连续混合调度）

评价指标：IFS+ CLIP Score
输出：同一 prompt 下三种方法的对比图 + 量化评测结果
"""

import os
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageFont
from scipy.linalg import sqrtm
from torchvision import transforms
from torchvision.models import inception_v3

warnings.filterwarnings("ignore")

# ============================================================
# 0. SEED
# ============================================================

def set_seed(seed: int):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def _load_font(size: int):
    """尝试加载系统字体，找不到就用 PIL 默认字体"""
    font_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSansBold.ttf",
        "/usr/share/fonts/truetype/ubuntu/Ubuntu-B.ttf",
    ]
    for p in font_paths:
        if os.path.exists(p):
            try:
                return ImageFont.truetype(p, size)
            except Exception:
                continue
    return ImageFont.load_default()


def _load_font_regular(size: int):
    font_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
        "/usr/share/fonts/truetype/ubuntu/Ubuntu-R.ttf",
    ]
    for p in font_paths:
        if os.path.exists(p):
            try:
                return ImageFont.truetype(p, size)
            except Exception:
                continue
    return ImageFont.load_default()

# ============================================================
# 1. IFS
# ============================================================

class ImageFidelityScorer:
    """
    衡量加速后生成的图片与 Baseline 原始图片的结构/语义差异。
    返回 PSNR (越高越好，衡量像素级差异) 和 Inception Cosine Similarity (越高越好，衡量语义级特征差异)。
    """
    def __init__(self, device: str = DEVICE):
        self.device = device
        self.model = inception_v3(pretrained=True, aux_logits=True)
        self.model.fc = nn.Identity() # 移除最后的分类层，获取 2048 维特征
        self.model = self.model.to(device).eval()
        self.preprocess = transforms.Compose([
            transforms.Resize((299, 299)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # 补上标准的归一化
        ])

    @torch.no_grad()
    def compute_metrics(self, img_baseline: Image.Image, img_generated: Image.Image) -> Dict[str, float]:
        # 1. 计算像素级 PSNR
        arr_b = np.array(img_baseline).astype(np.float32)
        arr_g = np.array(img_generated).astype(np.float32)
        mse = np.mean((arr_b - arr_g) ** 2)
        psnr = 20 * np.log10(255.0 / (np.sqrt(mse) + 1e-8)) if mse > 0 else 100.0

        # 2. 计算感知特征余弦相似度
        t_b = self.preprocess(img_baseline).unsqueeze(0).to(self.device)
        t_g = self.preprocess(img_generated).unsqueeze(0).to(self.device)

        feat_b = self.model(t_b)
        feat_g = self.model(t_g)

        cos_sim = F.cosine_similarity(feat_b, feat_g).item()

        return {"PSNR": float(psnr), "LPIPS_Proxy_Sim": float(cos_sim)}


# ============================================================
# 2. Standard CLIP Scorer
# ============================================================

class CLIPScorer:
    def __init__(self, device: str = DEVICE):
        self.device = device
        self.available = False
        try:
            from transformers import CLIPProcessor, CLIPModel
            self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
            self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
            self.available = True
            print("[CLIPScorer] Loaded (Using Cosine Similarity).")
        except Exception as e:
            print(f"[CLIPScorer] Not available: {e}. Using dummy.")

    @torch.no_grad()
    def score(self, image: Image.Image, text: str) -> float:
        if not self.available:
            return 0.25
        if image.mode != "RGB":
            image = image.convert("RGB")

        inputs = self.processor(text=[text], images=image, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # 获取标准的 normalized embeddings
        outputs = self.model(**inputs)
        image_embeds = outputs.image_embeds
        text_embeds = outputs.text_embeds

        image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        # 计算余弦相似度 [-1, 1]，通常较好的匹配在 0.25 ~ 0.35 以上
        cos_sim = (image_embeds @ text_embeds.T).item()
        return cos_sim

# ============================================================
# 3. 自适应缓存调度器
# ============================================================

@dataclass
class AdaptiveCacheConfig:
    use_piecewise: bool = True
    phase_boundaries: List[int] = field(default_factory=lambda: [35, 15])
    phase_intervals: List[int] = field(default_factory=lambda: [1, 3, 5])
    use_continuous: bool = False
    feature_change_threshold: float = 0.1
    min_cache_interval: int = 1
    max_cache_interval: int = 5
    cache_branch_id: int = 0
    cache_layer_id: int = 0
    total_steps: int = 50


class AdaptiveCacheScheduler:
    def __init__(self, config: AdaptiveCacheConfig):
        self.config = config
        self.steps_since_full = 0
        self.last_features: Optional[torch.Tensor] = None
        self.step_decisions: List[Dict] = []

    def should_compute_full(self, step, timestep, current_features=None) -> bool:
        if self.config.use_piecewise and not self.config.use_continuous:
            decision = self._piecewise_decision(step)
        elif self.config.use_continuous and not self.config.use_piecewise:
            decision = self._continuous_decision(step, current_features)
        else:
            decision = self._hybrid_decision(step, current_features)
        self.step_decisions.append({'step': step, 'timestep': timestep, 'full_compute': decision})
        return decision

    def _get_phase_interval(self, step):
        remaining = self.config.total_steps - step
        b, i = self.config.phase_boundaries, self.config.phase_intervals
        if remaining >= b[0]: return i[0]
        elif remaining >= b[1]: return i[1]
        else: return i[2]

    def _piecewise_decision(self, step):
        interval = self._get_phase_interval(step)
        if step == 0 or interval <= 1 or self.steps_since_full >= (interval - 1):
            self.steps_since_full = 0
            return True
        self.steps_since_full += 1
        return False

    def _continuous_decision(self, step, features):
        if features is None or self.last_features is None or step == 0:
            if features is not None: self.last_features = features.detach().clone()
            self.steps_since_full = 0
            return True
        change = torch.norm(features - self.last_features) / (torch.norm(self.last_features) + 1e-8)
        if change.item() > self.config.feature_change_threshold or self.steps_since_full >= self.config.max_cache_interval - 1:
            self.last_features = features.detach().clone()
            self.steps_since_full = 0
            return True
        self.steps_since_full += 1
        return False

    def _hybrid_decision(self, step, features):
        base_interval = self._get_phase_interval(step)
        if step == 0:
            if features is not None: self.last_features = features.detach().clone()
            self.steps_since_full = 0
            return True
        if features is not None and self.last_features is not None:
            change = torch.norm(features - self.last_features) / (torch.norm(self.last_features) + 1e-8)
            if change.item() > self.config.feature_change_threshold:
                self.last_features = features.detach().clone()
                self.steps_since_full = 0
                return True
        if base_interval <= 1 or self.steps_since_full >= (base_interval - 1):
            if features is not None: self.last_features = features.detach().clone()
            self.steps_since_full = 0
            return True
        if features is not None: self.last_features = features.detach().clone()
        self.steps_since_full += 1
        return False

    def reset(self):
        self.steps_since_full = 0
        self.last_features = None
        self.step_decisions = []

    def get_stats(self):
        if not self.step_decisions: return {}
        total = len(self.step_decisions)
        full = sum(1 for d in self.step_decisions if d['full_compute'])
        cached = total - full
        return {
            'total_steps': total, 'full_compute_steps': full,
            'cached_steps': cached,
            'cache_ratio': round(cached/total, 3) if total > 0 else 0,
            'theoretical_speedup': round(total/full, 2) if full > 0 else 1.0,
        }


# ============================================================
# 4. Timestep 预处理（对齐官方 get_time_embed）
# ============================================================

def _prepare_timestep(timestep, sample):
    """
    官方 UNet forward 中 get_time_embed() 的预处理逻辑。
    确保 timestep 是 shape=[batch_size] 的 1D 张量。
    不做这步会触发 assert len(timesteps.shape)==1 报错。
    """
    timesteps = timestep
    if not torch.is_tensor(timesteps):
        is_mps = sample.device.type == "mps"
        if isinstance(timestep, float):
            dtype = torch.float32 if is_mps else torch.float64
        else:
            dtype = torch.int32 if is_mps else torch.int64
        timesteps = torch.tensor([timesteps], dtype=dtype, device=sample.device)
    elif len(timesteps.shape) == 0:
        timesteps = timesteps[None].to(sample.device)
    timesteps = timesteps.expand(sample.shape[0])
    return timesteps


# ============================================================
# 5. 共享的 UNet 手动执行逻辑（消除重复代码）
# ============================================================

def _compute_embeddings(unet, sample, timestep, encoder_hidden_states, **kwargs):
    """计算 time_embedding + class_embedding + aug_embedding，处理 encoder_hidden_states"""
    timesteps = _prepare_timestep(timestep, sample)
    t_emb = unet.time_proj(timesteps)
    t_emb = t_emb.to(dtype=sample.dtype)
    emb = unet.time_embedding(t_emb)

    if hasattr(unet, 'class_embedding') and unet.class_embedding is not None:
        class_labels = kwargs.get('class_labels', None)
        if class_labels is not None:
            if unet.config.class_embed_type == "timestep":
                class_labels = unet.time_proj(class_labels)
                class_labels = class_labels.to(dtype=sample.dtype)
            class_emb = unet.class_embedding(class_labels).to(dtype=sample.dtype)
            if getattr(unet.config, 'class_embeddings_concat', False):
                emb = torch.cat([emb, class_emb], dim=-1)
            else:
                emb = emb + class_emb

    added_cond_kwargs = kwargs.get('added_cond_kwargs', None)
    if hasattr(unet, 'add_embedding') and added_cond_kwargs is not None:
        aug_emb = unet.get_aug_embed(emb=emb, encoder_hidden_states=encoder_hidden_states, added_cond_kwargs=added_cond_kwargs)
        if aug_emb is not None:
            emb = emb + aug_emb

    if hasattr(unet, 'time_embed_act') and unet.time_embed_act is not None:
        emb = unet.time_embed_act(emb)

    if hasattr(unet, 'encoder_hid_proj') and unet.encoder_hid_proj is not None:
        encoder_hidden_states = unet.process_encoder_hidden_states(
            encoder_hidden_states=encoder_hidden_states,
            added_cond_kwargs=added_cond_kwargs
        )

    return emb, encoder_hidden_states


def _run_encoder(unet, sample, emb, encoder_hidden_states):
    """运行 conv_in + down_blocks + mid_block"""
    sample = unet.conv_in(sample)
    down_block_res_samples = (sample,)

    for block in unet.down_blocks:
        if hasattr(block, "has_cross_attention") and block.has_cross_attention:
            sample, res = block(hidden_states=sample, temb=emb, encoder_hidden_states=encoder_hidden_states)
        else:
            sample, res = block(hidden_states=sample, temb=emb)
        down_block_res_samples += res

    if unet.mid_block is not None:
        if hasattr(unet.mid_block, "has_cross_attention") and unet.mid_block.has_cross_attention:
            sample = unet.mid_block(sample, emb, encoder_hidden_states=encoder_hidden_states)
        else:
            sample = unet.mid_block(sample, emb)

    return sample, down_block_res_samples


def _run_decoder(unet, sample, emb, down_block_res_samples, encoder_hidden_states):
    """运行 up_blocks + conv_out"""
    for block in unet.up_blocks:
        n = len(block.resnets)
        res = down_block_res_samples[-n:]
        down_block_res_samples = down_block_res_samples[:-n]
        if hasattr(block, "has_cross_attention") and block.has_cross_attention:
            sample = block(hidden_states=sample, temb=emb, res_hidden_states_tuple=res, encoder_hidden_states=encoder_hidden_states)
        else:
            sample = block(hidden_states=sample, temb=emb, res_hidden_states_tuple=res)

    if unet.conv_norm_out is not None:
        sample = unet.conv_norm_out(sample)
        sample = unet.conv_act(sample)
    sample = unet.conv_out(sample)
    return sample


# ============================================================
# 6. DeepCache Wrappers
# ============================================================

class DeepCacheUNetWrapper:
    """自适应 DeepCache：缓存 encoder+mid 输出，缓存步只跑 decoder"""

    def __init__(self, pipe, scheduler: Optional[AdaptiveCacheScheduler] = None):
        self.pipe = pipe
        self.unet = pipe.unet
        self.scheduler = scheduler
        self._cached_data: Optional[Dict] = None
        self._original_forward = self.unet.forward
        self._step_counter = 0

    def _run_full_and_cache(self, sample, timestep, encoder_hidden_states, **kwargs):
        from diffusers.models.unets.unet_2d_condition import UNet2DConditionOutput
        unet = self.unet
        emb, enc_hs = _compute_embeddings(unet, sample, timestep, encoder_hidden_states, **kwargs)
        mid_out, down_res = _run_encoder(unet, sample, emb, enc_hs)

        self._cached_data = {
            'down_block_res_samples': tuple(t.detach().clone() for t in down_res),
            'mid_output': mid_out.detach().clone(),
        }

        out = _run_decoder(unet, mid_out, emb, down_res, enc_hs)
        return UNet2DConditionOutput(sample=out)

    def _run_cached(self, sample, timestep, encoder_hidden_states, **kwargs):
        from diffusers.models.unets.unet_2d_condition import UNet2DConditionOutput
        unet = self.unet
        emb, enc_hs = _compute_embeddings(unet, sample, timestep, encoder_hidden_states, **kwargs)
        out = _run_decoder(unet, self._cached_data['mid_output'].clone(), emb,
                           self._cached_data['down_block_res_samples'], enc_hs)
        return UNet2DConditionOutput(sample=out)

    def enable(self):
        wrapper = self
        def cached_forward(sample, timestep, encoder_hidden_states=None, **kwargs):
            step = wrapper._step_counter
            ts = timestep.flatten()[0].item() if torch.is_tensor(timestep) else int(timestep)
            if wrapper.scheduler is not None:
                do_full = wrapper.scheduler.should_compute_full(step=step, timestep=ts, current_features=sample)
            else:
                do_full = (wrapper._cached_data is None)
            output = wrapper._run_full_and_cache(sample, timestep, encoder_hidden_states, **kwargs) if do_full \
                else wrapper._run_cached(sample, timestep, encoder_hidden_states, **kwargs)
            wrapper._step_counter += 1
            return output
        self.unet.forward = cached_forward
        self._step_counter = 0

    def disable(self):
        self.unet.forward = self._original_forward
        self._cached_data = None
        self._step_counter = 0
        if self.scheduler: self.scheduler.reset()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    @contextmanager
    def enabled(self):
        self.enable()
        try: yield self
        finally: self.disable()


class FixedIntervalDeepCache:
    """标准 DeepCache：固定间隔"""

    def __init__(self, pipe, cache_interval: int = 3):
        self.pipe = pipe
        self.unet = pipe.unet
        self.cache_interval = cache_interval
        self._cached_data: Optional[Dict] = None
        self._original_forward = self.unet.forward
        self._step_counter = 0

    def _run_full_and_cache(self, sample, timestep, encoder_hidden_states, **kwargs):
        from diffusers.models.unets.unet_2d_condition import UNet2DConditionOutput
        unet = self.unet
        emb, enc_hs = _compute_embeddings(unet, sample, timestep, encoder_hidden_states, **kwargs)
        mid_out, down_res = _run_encoder(unet, sample, emb, enc_hs)
        self._cached_data = {
            'down_block_res_samples': tuple(t.detach().clone() for t in down_res),
            'mid_output': mid_out.detach().clone(),
        }
        out = _run_decoder(unet, mid_out, emb, down_res, enc_hs)
        return UNet2DConditionOutput(sample=out)

    def _run_cached(self, sample, timestep, encoder_hidden_states, **kwargs):
        from diffusers.models.unets.unet_2d_condition import UNet2DConditionOutput
        unet = self.unet
        emb, enc_hs = _compute_embeddings(unet, sample, timestep, encoder_hidden_states, **kwargs)
        out = _run_decoder(unet, self._cached_data['mid_output'].clone(), emb,
                           self._cached_data['down_block_res_samples'], enc_hs)
        return UNet2DConditionOutput(sample=out)

    def enable(self):
        wrapper = self
        def cached_forward(sample, timestep, encoder_hidden_states=None, **kwargs):
            step = wrapper._step_counter
            do_full = (step % wrapper.cache_interval == 0) or (wrapper._cached_data is None)
            output = wrapper._run_full_and_cache(sample, timestep, encoder_hidden_states, **kwargs) if do_full \
                else wrapper._run_cached(sample, timestep, encoder_hidden_states, **kwargs)
            wrapper._step_counter += 1
            return output
        self.unet.forward = cached_forward
        self._step_counter = 0

    def disable(self):
        self.unet.forward = self._original_forward
        self._cached_data = None
        self._step_counter = 0
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    @contextmanager
    def enabled(self):
        self.enable()
        try: yield self
        finally: self.disable()


# ============================================================
# 7. 对比图生成（核心输出）
# ============================================================

def create_comparison_image(
    img_baseline: Image.Image,
    img_deepcache: Image.Image,
    img_adaptive: Image.Image,
    prompt: str,
    times: Tuple[float, float, float],
    fids: Tuple[float, float, float],
    clips: Tuple[float, float, float],
) -> Image.Image:
    """
    生成一张三列对比图：DDIM Baseline | DeepCache | Adaptive DeepCache
    每列下方标注: 生成时间、加速比、FID、CLIP Score
    底部标注 prompt
    """
    W, H = img_baseline.size
    pad = 20
    header_h = 60
    metrics_h = 80
    prompt_h = 40
    total_w = W * 3 + pad * 4
    total_h = header_h + H + metrics_h + prompt_h + pad * 2

    canvas = Image.new("RGB", (total_w, total_h), "#FFFFFF")
    draw = ImageDraw.Draw(canvas)

    font_title = _load_font(18)
    font_sub = _load_font_regular(13)
    font_metric = _load_font_regular(12)
    font_prompt = _load_font_regular(11)

    # 标题栏背景
    draw.rectangle([(0, 0), (total_w, header_h)], fill="#2C3E50")

    headers = [
        ("DDIM Baseline", "(No Cache, Full Inference)"),
        ("DeepCache (Fixed)", f"(interval=fixed, every N steps)"),
        ("Adaptive DeepCache", "(Ours: piecewise + continuous)"),
    ]
    colors = ["#ECF0F1", "#ECF0F1", "#F1C40F"]  # 第三个用金色高亮
    images = [img_baseline, img_deepcache, img_adaptive]

    for i, ((title, sub), color, img) in enumerate(zip(headers, colors, images)):
        x_start = pad + i * (W + pad)
        cx = x_start + W // 2

        # 标题文字
        draw.text((cx, 12), title, fill=color, font=font_title, anchor="mt")
        draw.text((cx, 35), sub, fill="#95A5A6", font=font_sub, anchor="mt")

        # 图片
        y_img = header_h + pad // 2
        canvas.paste(img.resize((W, H)), (x_start, y_img))

        # 图片边框
        draw.rectangle(
            [(x_start-1, y_img-1), (x_start+W, y_img+H)],
            outline="#BDC3C7", width=1
        )

        # 指标区域
        y_m = y_img + H + 8
        sp = times[0] / max(times[i], 1e-6)

        lines = [
            f"⏱ Time: {times[i]:.2f}s",
            f"⚡ Speedup: {sp:.2f}x" if i > 0 else "⚡ Speedup: 1.00x (baseline)",
            f"📊 PSNR: {fids[i]:.1f} dB" if i > 0 else "📊 PSNR: — (reference)",
            f"🎯 CLIP: {clips[i]:.4f}",
        ]
        for j, line in enumerate(lines):
            draw.text((x_start + 8, y_m + j * 17), line, fill="#2C3E50", font=font_metric)

    # 底部 prompt
    y_prompt = total_h - prompt_h + 5
    draw.rectangle([(0, y_prompt - 5), (total_w, total_h)], fill="#ECF0F1")
    prompt_display = prompt if len(prompt) <= 100 else prompt[:97] + "..."
    draw.text((total_w // 2, y_prompt + 8), f'Prompt: "{prompt_display}"',
              fill="#7F8C8D", font=font_prompt, anchor="mt")

    return canvas


# ============================================================
# 8. 主实验 Pipeline
# ============================================================

class AdaptiveDeepCacheExperiment:
    def __init__(self, model_id="runwayml/stable-diffusion-v1-5", device=DEVICE):
        self.device = device
        print(f"[Experiment] Device: {self.device}")

        from diffusers import StableDiffusionPipeline, DDIMScheduler

        print(f"[Experiment] Loading: {model_id} ...")
        self.pipe = StableDiffusionPipeline.from_pretrained(
            model_id, torch_dtype=DTYPE,
            safety_checker=None, requires_safety_checker=False
        ).to(self.device)
        self.pipe.scheduler = DDIMScheduler.from_config(self.pipe.scheduler.config)

        print("[Experiment] Loading evaluators...")
        self.fidelity_scorer = ImageFidelityScorer(device)
        self.clip_scorer = CLIPScorer(device)
        print("[Experiment] Ready.\n")

    def generate_ddim_baseline(self, prompt, num_steps=50, seed=42):
        set_seed(seed)
        start = time.time()
        with torch.inference_mode():
            output = self.pipe(prompt, num_inference_steps=num_steps, output_type="pil")
        return output.images[0], time.time() - start

    def generate_deepcache(self, prompt, num_steps=50, seed=42, cache_interval=3):
        set_seed(seed)
        helper = FixedIntervalDeepCache(self.pipe, cache_interval=cache_interval)
        with helper.enabled():
            start = time.time()
            with torch.inference_mode():
                output = self.pipe(prompt, num_inference_steps=num_steps, output_type="pil")
            elapsed = time.time() - start
        return output.images[0], elapsed

    def generate_adaptive_deepcache(self, prompt, num_steps=50, seed=42, config=None):
        set_seed(seed)
        if config is None:
            config = AdaptiveCacheConfig(
                use_piecewise=True, use_continuous=True,
                phase_boundaries=[35, 15], phase_intervals=[1, 3, 5],
                feature_change_threshold=0.12, max_cache_interval=5,
                total_steps=num_steps
            )
        else:
            config.total_steps = num_steps
        scheduler = AdaptiveCacheScheduler(config)
        helper = DeepCacheUNetWrapper(self.pipe, scheduler=scheduler)
        with helper.enabled():
            start = time.time()
            with torch.inference_mode():
                output = self.pipe(prompt, num_inference_steps=num_steps, output_type="pil")
            elapsed = time.time() - start
            stats = scheduler.get_stats()
        return output.images[0], elapsed, stats

    def compare_single_prompt(
        self, prompt, num_steps=50, seed=42,
        deepcache_interval=3, adaptive_config=None,
        save_dir="results", display_in_notebook=True
    ) -> Dict:
        os.makedirs(save_dir, exist_ok=True)

        print(f'  Prompt: "{prompt}"')
        print(f"  Steps: {num_steps}, Seed: {seed}\n")

        # ── 生成三张图 ──
        print("  [1/3] DDIM Baseline ...")
        img_bl, t_bl = self.generate_ddim_baseline(prompt, num_steps, seed)
        print(f"        Time: {t_bl:.2f}s")

        print(f"  [2/3] DeepCache (interval={deepcache_interval}) ...")
        img_dc, t_dc = self.generate_deepcache(prompt, num_steps, seed, deepcache_interval)
        sp_dc = t_bl / max(t_dc, 1e-6)
        print(f"        Time: {t_dc:.2f}s  (Speedup: {sp_dc:.2f}x)")

        print("  [3/3] Adaptive DeepCache ...")
        img_ad, t_ad, ad_stats = self.generate_adaptive_deepcache(prompt, num_steps, seed, adaptive_config)
        sp_ad = t_bl / max(t_ad, 1e-6)
        print(f"        Time: {t_ad:.2f}s  (Speedup: {sp_ad:.2f}x)")
        print(f"        Schedule: {ad_stats}")

        # ── 评测 ──
        print("\n  Computing metrics ...")
        # 调用新的 fidelity_scorer，返回一个字典
        metrics_dc = self.fidelity_scorer.compute_metrics(img_bl, img_dc)
        metrics_ad = self.fidelity_scorer.compute_metrics(img_bl, img_ad)



        psnr_dc = metrics_dc["PSNR"]
        sim_dc  = metrics_dc["LPIPS_Proxy_Sim"]   # 新增
        psnr_ad = metrics_ad["PSNR"]
        sim_ad  = metrics_ad["LPIPS_Proxy_Sim"]   # 新增

        # CLIP 保持不变（但返回值现在是余弦相似度）
        clip_bl = self.clip_scorer.score(img_bl, prompt)
        clip_dc = self.clip_scorer.score(img_dc, prompt)
        clip_ad = self.clip_scorer.score(img_ad, prompt)

        print(f"  PSNR (DC vs BL):  {psnr_dc:.2f} dB")
        print(f"  PSNR (ADC vs BL): {psnr_ad:.2f} dB")
        print(f"  Feat Sim (DC):    {sim_dc:.4f}")
        print(f"  Feat Sim (ADC):   {sim_ad:.4f}")
        print(f"  CLIP (BL):  {clip_bl:.4f}")
        print(f"  CLIP (DC):  {clip_dc:.4f}")
        print(f"  CLIP (ADC): {clip_ad:.4f}")

        # ── 生成对比图 ──
        comparison = create_comparison_image(
            img_bl, img_dc, img_ad,
            prompt=prompt,
            times=(t_bl, t_dc, t_ad),
            fids=(0.0, psnr_dc, psnr_ad),
            clips=(clip_bl, clip_dc, clip_ad),
        )

        # 保存
        safe = "".join(c if c.isalnum() or c in "_ " else "" for c in prompt[:40]).strip().replace(" ", "_")
        path = os.path.join(save_dir, f"comparison_{safe}_s{seed}.png")
        comparison.save(path, quality=95)
        print(f"\n  ✅ Comparison saved → {path}")

        # 在 Notebook 中直接显示
        if display_in_notebook:
            try:
                from IPython.display import display as ipy_display
                print(f"\n  === Comparison: {prompt} ===")
                ipy_display(comparison)
            except ImportError:
                pass

        return {
            'prompt': prompt, 'seed': seed, 'num_steps': num_steps,
            'comparison_image': comparison,
            'comparison_path': path,
            'images': {'baseline': img_bl, 'deepcache': img_dc, 'adaptive': img_ad},
            'times': {'baseline': t_bl, 'deepcache': t_dc, 'adaptive': t_ad},
            'speedups': {'deepcache': sp_dc, 'adaptive': sp_ad},
            'fid': {'deepcache_vs_baseline': psnr_dc, 'adaptive_vs_baseline': psnr_ad},
            'clip': {'baseline': clip_bl, 'deepcache': clip_dc, 'adaptive': clip_ad},
            'adaptive_stats': ad_stats,
        }

    def run_full_benchmark(
        self, prompts, seeds=None, num_steps=50,
        deepcache_interval=3, adaptive_config=None,
        save_dir="benchmark_results", display_in_notebook=True
    ) -> Dict:
        if seeds is None: seeds = [42, 123, 456]
        os.makedirs(save_dir, exist_ok=True)
        all_results = []
        total = len(prompts) * len(seeds)
        idx = 0

        for prompt in prompts:
            for seed in seeds:
                idx += 1
                print(f"\n{'='*60}")
                print(f"  Run {idx}/{total}")
                print(f"{'='*60}")
                result = self.compare_single_prompt(
                    prompt=prompt, num_steps=num_steps, seed=seed,
                    deepcache_interval=deepcache_interval,
                    adaptive_config=adaptive_config, save_dir=save_dir,
                    display_in_notebook=display_in_notebook,
                )
                all_results.append(result)

        summary = self._compute_summary(all_results)
        self._print_summary(summary)
        self._save_summary_table(summary, all_results, save_dir)
        return {'individual_results': all_results, 'summary': summary}

    def _compute_summary(self, results):
        n = len(results)
        def s(vals): return (float(np.mean(vals)), float(np.std(vals)))
        return {
            'n_runs': n,
            'time': {k: s([r['times'][k] for r in results]) for k in ['baseline','deepcache','adaptive']},
            'speedup': {k: s([r['speedups'][k] for r in results]) for k in ['deepcache','adaptive']},
            'fid': {
                'deepcache': s([r['fid']['deepcache_vs_baseline'] for r in results]),
                'adaptive': s([r['fid']['adaptive_vs_baseline'] for r in results]),
            },
            'clip': {k: s([r['clip'][k] for r in results]) for k in ['baseline','deepcache','adaptive']},
        }

    def _print_summary(self, s):
        print("\n" + "="*75)
        print("  BENCHMARK SUMMARY")
        print("="*75)
        print(f"  Total runs: {s['n_runs']}\n")
        print(f"  {'Method':<25} {'Time (s)':<18} {'Speedup':<12} {'FID ↓':<15} {'CLIP ↑':<15}")
        print(f"  {'-'*85}")

        t = s['time']['baseline']; c = s['clip']['baseline']
        print(f"  {'DDIM Baseline':<25} {t[0]:.2f}±{t[1]:.2f}{'':>6} {'1.00x':<12} {'—':<15} {c[0]:.3f}±{c[1]:.3f}")

        for m, name in [('deepcache','DeepCache (fixed)'),('adaptive','Adaptive DC (ours)')]:
            t = s['time'][m]; sp = s['speedup'][m]; f = s['fid'][m]; c = s['clip'][m]
            print(f"  {name:<25} {t[0]:.2f}±{t[1]:.2f}{'':>6} {sp[0]:.2f}x±{sp[1]:.2f}  {f[0]:.1f}±{f[1]:.1f}{'':>5} {c[0]:.3f}±{c[1]:.3f}")

        print("="*75)
        print("  PSNR  = HIGHER is better    CLIP ↑ = higher is better")
        print("="*75)

    def _save_summary_table(self, summary, results, save_dir):
        path = os.path.join(save_dir, "benchmark_summary.txt")
        with open(path, 'w', encoding='utf-8') as f:
            f.write("Adaptive DeepCache Benchmark\n" + "="*50 + "\n\n")
            for i, r in enumerate(results):
                f.write(f"Run {i+1}: \"{r['prompt'][:50]}\" seed={r['seed']}\n")
                f.write(f"  Time: BL={r['times']['baseline']:.2f}s DC={r['times']['deepcache']:.2f}s ADC={r['times']['adaptive']:.2f}s\n")
                f.write(f"  PSNR:  DC={r['fid']['deepcache_vs_baseline']:.2f} ADC={r['fid']['adaptive_vs_baseline']:.2f}\n")
                f.write(f"  CLIP: BL={r['clip']['baseline']:.4f} DC={r['clip']['deepcache']:.4f} ADC={r['clip']['adaptive']:.4f}\n")
                f.write(f"  Adaptive: {r['adaptive_stats']}\n\n")
        print(f"  Summary saved → {path}")


# ============================================================
# 9. 入口
# ============================================================

def main():
    print("="*60)
    print("  Adaptive DeepCache Experiment")
    print("  DDIM vs DeepCache vs Adaptive DeepCache")
    print("="*60 + "\n")

    adaptive_config = AdaptiveCacheConfig(
        use_piecewise=True, use_continuous=True,
        phase_boundaries=[35, 15], phase_intervals=[1, 3, 5],
        feature_change_threshold=0.12, max_cache_interval=5,
        total_steps=50,
    )

    prompts = [
        "a beautiful sunset over hill, photorealistic, 8k",
        "A long time ago, in a galaxy far away...",
        "an astronaut lost in the wide space",
        "clone war in star wars, a Jedi against strom fighters, detailed",
    ]

    experiment = AdaptiveDeepCacheExperiment(
        model_id="runwayml/stable-diffusion-v1-5",
        device=DEVICE
    )

    results = experiment.run_full_benchmark(
        prompts=prompts,
        seeds=[42, 123, 456],
        num_steps=50,
        deepcache_interval=3,
        adaptive_config=adaptive_config,
        save_dir="adaptive_deepcache_results",
        display_in_notebook=True,  # Colab 中直接显示对比图
    )

    print("\n\nDone! Check 'adaptive_deepcache_results/' for outputs.")


if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.

# **DEEP CACHE PACK**

In [ ]:
!pip install DeepCache

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 15.9 MB/s eta 0:00:00


In [ ]:
"""
自适应 DeepCache 实验框架（使用官方 DeepCacheSDHelper）
======================================================
三种方法对比：
  1. DDIM Baseline（无缓存，完整推理）
  2. DeepCache（官方 DeepCacheSDHelper，固定缓存间隔）
  3. Adaptive DeepCache（基于官方 hook 机制，自适应调度）

评价指标：FID + CLIP Score
输出：对比图 + 量化评测
"""

# ── 安装依赖（Colab 首次运行）──
# !pip install -q diffusers transformers accelerate DeepCache

import os
import time
import warnings
from contextlib import contextmanager
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict, Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
from scipy.linalg import sqrtm
from torchvision import transforms
from torchvision.models import inception_v3

warnings.filterwarnings("ignore")


# ============================================================
# 0. 全局工具
# ============================================================

def set_seed(seed: int):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32


def _load_font(size):
    for p in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
    ]:
        if os.path.exists(p):
            try: return ImageFont.truetype(p, size)
            except: continue
    return ImageFont.load_default()

def _load_font_regular(size):
    for p in [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ]:
        if os.path.exists(p):
            try: return ImageFont.truetype(p, size)
            except: continue
    return ImageFont.load_default()


# ============================================================
# 1. FID 计算器
# ============================================================

class FIDCalculator:
    def __init__(self, device=DEVICE):
        self.device = device
        self.model = inception_v3(pretrained=True, aux_logits=True)
        self.model.aux_logits = False
        self.model.AuxLogits = None
        self.model.fc = nn.Identity()
        self.model = self.model.to(device).eval()
        self.preprocess = transforms.Compose([
            transforms.Resize((299, 299)), transforms.ToTensor(),
        ])

    @torch.no_grad()
    def extract_features(self, images):
        feats = []
        for img in images:
            if img.mode != "RGB": img = img.convert("RGB")
            t = self.preprocess(img).unsqueeze(0).to(self.device)
            feats.append(self.model(t).cpu().numpy())
        return np.concatenate(feats, axis=0)

    def calculate_fid(self, imgs_ref, imgs_gen):
        if len(imgs_ref) < 2 or len(imgs_gen) < 2:
            warnings.warn("FID with <2 images degrades to feature L2 distance.", stacklevel=2)
        fr, fg = self.extract_features(imgs_ref), self.extract_features(imgs_gen)
        mr, mg = fr.mean(0), fg.mean(0)
        sr = np.zeros((fr.shape[1],)*2) if fr.shape[0]==1 else np.cov(fr, rowvar=False)
        sg = np.zeros((fg.shape[1],)*2) if fg.shape[0]==1 else np.cov(fg, rowvar=False)
        diff = mr - mg
        if np.allclose(sr, 0) and np.allclose(sg, 0):
            return float(np.sum(diff**2))
        eps = 1e-6
        sr += np.eye(sr.shape[0])*eps; sg += np.eye(sg.shape[0])*eps
        cm, _ = sqrtm(sr @ sg, disp=False)
        if np.iscomplexobj(cm): cm = cm.real
        np.fill_diagonal(cm, np.maximum(np.diagonal(cm), 0))
        return float(max(np.sum(diff**2) + np.trace(sr + sg - 2*cm), 0))


# ============================================================
# 2. CLIP 评分器
# ============================================================

class CLIPScorer:
    def __init__(self, device=DEVICE):
        self.device = device; self.available = False
        try:
            from transformers import CLIPProcessor, CLIPModel
            self.model = CLIPModel.from_pretrained("openai/clip-vit-base-patch16").to(device).eval()
            self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
            self.available = True
            print("[CLIPScorer] Loaded.")
        except Exception as e:
            print(f"[CLIPScorer] Not available: {e}")

    @torch.no_grad()
    def score(self, image, text):
        if not self.available: return 0.25
        if image.mode != "RGB": image = image.convert("RGB")
        inputs = self.processor(text=[text], images=image, return_tensors="pt", padding=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        logits = self.model(**inputs).logits_per_image.item() / 100.0
        return max(0.0, min(1.0, logits))


# ============================================================
# 3. 自适应缓存调度器（核心创新，替换官方的固定 is_skip_step）
# ============================================================

@dataclass
class AdaptiveCacheConfig:
    """
    分段 + 连续自适应配置

    phase_intervals 含义：
      interval=1 → 每步完整计算（不缓存）
      interval=3 → 每 3 步做 1 次完整计算
    """
    use_piecewise: bool = True
    phase_boundaries: List[int] = field(default_factory=lambda: [35, 15])
    phase_intervals: List[int] = field(default_factory=lambda: [1, 3, 5])

    use_continuous: bool = False
    feature_change_threshold: float = 0.1
    max_cache_interval: int = 5

    cache_interval: int = 3        # 作为 fallback 的固定间隔
    cache_branch_id: int = 0       # 与官方一致
    total_steps: int = 50


class AdaptiveCacheScheduler:
    """
    自适应调度器：替换 DeepCacheSDHelper 的固定 is_skip_step 判断

    核心思想：不同去噪阶段使用不同缓存间隔
      早期（高噪声）→ interval 小（密集计算）
      后期（低噪声）→ interval 大（激进缓存）
    """

    def __init__(self, config: AdaptiveCacheConfig):
        self.config = config
        self.steps_since_full = 0
        self.step_decisions: List[Dict] = []

    def should_skip(self, cur_step: int) -> bool:
        """
        返回 True = 使用缓存（跳过计算）
        返回 False = 完整计算（不跳过）
        """
        if self.config.use_piecewise:
            interval = self._get_phase_interval(cur_step)
        else:
            interval = self.config.cache_interval

        # 第 0 步必须完整计算
        if cur_step == 0 or interval <= 1:
            do_full = True
        elif self.steps_since_full >= (interval - 1):
            do_full = True
        else:
            do_full = False

        if do_full:
            self.steps_since_full = 0
        else:
            self.steps_since_full += 1

        self.step_decisions.append({
            'step': cur_step, 'full_compute': do_full, 'interval': interval
        })
        return not do_full  # skip = not full

    def _get_phase_interval(self, step):
        remaining = self.config.total_steps - step
        b, i = self.config.phase_boundaries, self.config.phase_intervals
        if remaining >= b[0]: return i[0]
        elif remaining >= b[1]: return i[1]
        else: return i[2]

    def reset(self):
        self.steps_since_full = 0
        self.step_decisions = []

    def get_stats(self):
        if not self.step_decisions: return {}
        total = len(self.step_decisions)
        full = sum(1 for d in self.step_decisions if d['full_compute'])
        return {
            'total_steps': total, 'full_compute_steps': full,
            'cached_steps': total-full,
            'cache_ratio': round((total-full)/total, 3),
            'theoretical_speedup': round(total/full, 2) if full else 1.0,
        }


# ============================================================
# 4. 自适应 DeepCache Helper
#    基于官方 DeepCacheSDHelper 的 hook 机制
#    仅替换 is_skip_step 为自适应调度
# ============================================================

class AdaptiveDeepCacheSDHelper:
    """
    基于官方 DeepCacheSDHelper 相同的子模块 hook 机制，
    但用自适应调度器替换固定间隔的 is_skip_step。

    官方做法：每个子模块 forward 被 wrap，根据 (step % interval == 0) 判断跳不跳
    我们做法：每个子模块 forward 被 wrap，根据 AdaptiveCacheScheduler 判断跳不跳

    优势：
      - 不拆解 UNet forward（不会触发 timestep 维度问题）
      - 与 diffusers 版本无关（hook 在子模块级别）
      - cache_branch_id 控制缓存深度
    """

    def __init__(self, pipe, scheduler: AdaptiveCacheScheduler):
        self.pipe = pipe
        self.scheduler = scheduler
        self.function_dict = {}
        self.cached_output = {}
        self.cur_timestep = 0
        self.start_timestep = None
        self._step_decided = {}  # 每步只调度一次

    def set_params(self, cache_branch_id=0):
        cache_layer_id = cache_branch_id % 3
        cache_block_id = cache_branch_id // 3
        self.params = {
            'cache_layer_id': cache_layer_id,
            'cache_block_id': cache_block_id,
        }

    def is_skip_step(self, block_i, layer_i, blocktype="down"):
        """
        与官方逻辑相同：根据 cache_branch_id 决定哪些层在跳过步时使用缓存。
        区别：跳不跳由 AdaptiveCacheScheduler 决定（而非固定间隔）。
        """
        self.start_timestep = self.cur_timestep if self.start_timestep is None else self.start_timestep

        # 每个 timestep 只调用一次调度器
        if self.cur_timestep not in self._step_decided:
            self._step_decided[self.cur_timestep] = self.scheduler.should_skip(self.cur_timestep)
        should_skip = self._step_decided[self.cur_timestep]

        if not should_skip:
            return False  # 完整计算步：所有层正常运行

        # 跳过步：根据 cache_branch_id 决定哪些层用缓存
        cache_layer_id = self.params['cache_layer_id']
        cache_block_id = self.params['cache_block_id']

        if block_i > cache_block_id or blocktype == 'mid':
            return True
        if block_i < cache_block_id:
            return False
        return layer_i >= cache_layer_id if blocktype == 'down' else layer_i > cache_layer_id

    # ── 以下与官方 DeepCacheSDHelper 完全一致 ──

    def wrap_unet_forward(self):
        self.function_dict['unet_forward'] = self.pipe.unet.forward
        helper = self
        def wrapped_forward(*args, **kwargs):
            helper.cur_timestep = list(helper.pipe.scheduler.timesteps).index(args[1].item())
            return helper.function_dict['unet_forward'](*args, **kwargs)
        self.pipe.unet.forward = wrapped_forward

    def wrap_block_forward(self, block, block_name, block_i, layer_i, blocktype="down"):
        self.function_dict[(blocktype, block_name, block_i, layer_i)] = block.forward
        helper = self
        def wrapped_forward(*args, **kwargs):
            skip = helper.is_skip_step(block_i, layer_i, blocktype)
            key = (blocktype, block_name, block_i, layer_i)
            if skip and key in helper.cached_output:
                return helper.cached_output[key]
            result = helper.function_dict[key](*args, **kwargs)
            helper.cached_output[key] = result
            return result
        block.forward = wrapped_forward

    def wrap_modules(self):
        self.wrap_unet_forward()
        for block_i, block in enumerate(self.pipe.unet.down_blocks):
            for layer_i, attn in enumerate(getattr(block, "attentions", [])):
                self.wrap_block_forward(attn, "attentions", block_i, layer_i)
            for layer_i, resnet in enumerate(getattr(block, "resnets", [])):
                self.wrap_block_forward(resnet, "resnet", block_i, layer_i)
            for ds in (getattr(block, "downsamplers", []) or []):
                self.wrap_block_forward(ds, "downsampler", block_i, len(getattr(block, "resnets", [])))
            self.wrap_block_forward(block, "block", block_i, 0, blocktype="down")
        self.wrap_block_forward(self.pipe.unet.mid_block, "mid_block", 0, 0, blocktype="mid")
        n = len(self.pipe.unet.up_blocks)
        for block_i, block in enumerate(self.pipe.unet.up_blocks):
            ln = len(getattr(block, "resnets", []))
            for layer_i, attn in enumerate(getattr(block, "attentions", [])):
                self.wrap_block_forward(attn, "attentions", n-block_i-1, ln-layer_i-1, "up")
            for layer_i, resnet in enumerate(getattr(block, "resnets", [])):
                self.wrap_block_forward(resnet, "resnet", n-block_i-1, ln-layer_i-1, "up")
            for us in (getattr(block, "upsamplers", []) or []):
                self.wrap_block_forward(us, "upsampler", n-block_i-1, 0, "up")
            self.wrap_block_forward(block, "block", n-block_i-1, 0, blocktype="up")

    def unwrap_modules(self):
        self.pipe.unet.forward = self.function_dict['unet_forward']
        for block_i, block in enumerate(self.pipe.unet.down_blocks):
            for layer_i, attn in enumerate(getattr(block, "attentions", [])):
                attn.forward = self.function_dict[("down", "attentions", block_i, layer_i)]
            for layer_i, resnet in enumerate(getattr(block, "resnets", [])):
                resnet.forward = self.function_dict[("down", "resnet", block_i, layer_i)]
            for ds in (getattr(block, "downsamplers", []) or []):
                ds.forward = self.function_dict[("down", "downsampler", block_i, len(getattr(block, "resnets", [])))]
            block.forward = self.function_dict[("down", "block", block_i, 0)]
        self.pipe.unet.mid_block.forward = self.function_dict[("mid", "mid_block", 0, 0)]
        n = len(self.pipe.unet.up_blocks)
        for block_i, block in enumerate(self.pipe.unet.up_blocks):
            ln = len(getattr(block, "resnets", []))
            for layer_i, attn in enumerate(getattr(block, "attentions", [])):
                attn.forward = self.function_dict[("up", "attentions", n-block_i-1, ln-layer_i-1)]
            for layer_i, resnet in enumerate(getattr(block, "resnets", [])):
                resnet.forward = self.function_dict[("up", "resnet", n-block_i-1, ln-layer_i-1)]
            for us in (getattr(block, "upsamplers", []) or []):
                us.forward = self.function_dict[("up", "upsampler", n-block_i-1, 0)]
            block.forward = self.function_dict[("up", "block", n-block_i-1, 0)]

    def enable(self):
        self.reset_states()
        self.wrap_modules()

    def disable(self):
        self.unwrap_modules()
        self.reset_states()

    def reset_states(self):
        self.cur_timestep = 0
        self.function_dict = {}
        self.cached_output = {}
        self.start_timestep = None
        self._step_decided = {}
        self.scheduler.reset()

    @contextmanager
    def enabled(self):
        self.enable()
        try: yield self
        finally: self.disable()


# ============================================================
# 5. 对比图生成
# ============================================================

def create_comparison_image(img_bl, img_dc, img_ad, prompt, times, fids, clips):
    W, H = img_bl.size
    pad = 20; hdr = 60; met = 80; pmt = 40
    tw = W*3 + pad*4; th = hdr + H + met + pmt + pad*2
    canvas = Image.new("RGB", (tw, th), "#FFFFFF")
    draw = ImageDraw.Draw(canvas)
    ft, fs, fm, fp = _load_font(18), _load_font_regular(13), _load_font_regular(12), _load_font_regular(11)

    draw.rectangle([(0,0),(tw,hdr)], fill="#2C3E50")
    headers = [
        ("DDIM Baseline", "(No Cache)"),
        ("DeepCache (Official)", "(Fixed interval)"),
        ("Adaptive DeepCache", "(Ours: piecewise schedule)"),
    ]
    colors = ["#ECF0F1","#ECF0F1","#F1C40F"]
    imgs = [img_bl, img_dc, img_ad]

    for i, ((t, s), c, im) in enumerate(zip(headers, colors, imgs)):
        x = pad + i*(W+pad); cx = x + W//2
        draw.text((cx,12), t, fill=c, font=ft, anchor="mt")
        draw.text((cx,35), s, fill="#95A5A6", font=fs, anchor="mt")
        yi = hdr + pad//2
        canvas.paste(im.resize((W,H)), (x, yi))
        draw.rectangle([(x-1,yi-1),(x+W,yi+H)], outline="#BDC3C7", width=1)
        ym = yi + H + 8
        sp = times[0]/max(times[i],1e-6)
        lines = [
            f"⏱ Time: {times[i]:.2f}s",
            f"⚡ Speedup: {sp:.2f}x" if i>0 else "⚡ Speedup: 1.00x (baseline)",
            f"📊 FID: {fids[i]:.1f}" if i>0 else "📊 FID: — (reference)",
            f"🎯 CLIP: {clips[i]:.4f}",
        ]
        for j, ln in enumerate(lines):
            draw.text((x+8, ym+j*17), ln, fill="#2C3E50", font=fm)

    yp = th - pmt + 5
    draw.rectangle([(0,yp-5),(tw,th)], fill="#ECF0F1")
    pd = prompt if len(prompt)<=100 else prompt[:97]+"..."
    draw.text((tw//2, yp+8), f'Prompt: "{pd}"', fill="#7F8C8D", font=fp, anchor="mt")
    return canvas


# ============================================================
# 6. 主实验 Pipeline
# ============================================================

class AdaptiveDeepCacheExperiment:
    def __init__(self, model_id="runwayml/stable-diffusion-v1-5", device=DEVICE):
        self.device = device
        print(f"[Experiment] Device: {self.device}")

        from diffusers import StableDiffusionPipeline, DDIMScheduler

        print(f"[Experiment] Loading: {model_id} ...")
        self.pipe = StableDiffusionPipeline.from_pretrained(
            model_id, torch_dtype=DTYPE,
            safety_checker=None, requires_safety_checker=False
        ).to(self.device)
        self.pipe.scheduler = DDIMScheduler.from_config(self.pipe.scheduler.config)

        print("[Experiment] Loading evaluators...")
        self.fid_calc = FIDCalculator(device)
        self.clip_scorer = CLIPScorer(device)
        print("[Experiment] Ready.\n")

    # ── 方法 1: DDIM Baseline ──

    def generate_ddim(self, prompt, num_steps=50, seed=42):
        set_seed(seed)
        t0 = time.time()
        with torch.inference_mode():
            out = self.pipe(prompt, num_inference_steps=num_steps, output_type="pil")
        return out.images[0], time.time()-t0

    # ── 方法 2: 官方 DeepCache ──

    def generate_deepcache(self, prompt, num_steps=50, seed=42,
                           cache_interval=3, cache_branch_id=0):
        from DeepCache import DeepCacheSDHelper

        set_seed(seed)
        helper = DeepCacheSDHelper(pipe=self.pipe)
        helper.set_params(cache_interval=cache_interval, cache_branch_id=cache_branch_id)
        helper.enable()
        try:
            t0 = time.time()
            with torch.inference_mode():
                out = self.pipe(prompt, num_inference_steps=num_steps, output_type="pil")
            elapsed = time.time()-t0
        finally:
            helper.disable()
        return out.images[0], elapsed

    # ── 方法 3: 自适应 DeepCache ──

    def generate_adaptive(self, prompt, num_steps=50, seed=42,
                          config: Optional[AdaptiveCacheConfig] = None):
        set_seed(seed)
        if config is None:
            config = AdaptiveCacheConfig(
                use_piecewise=True, use_continuous=False,
                phase_boundaries=[35, 15], phase_intervals=[1, 3, 5],
                cache_branch_id=0, total_steps=num_steps,
            )
        else:
            config.total_steps = num_steps

        scheduler = AdaptiveCacheScheduler(config)
        helper = AdaptiveDeepCacheSDHelper(self.pipe, scheduler)
        helper.set_params(cache_branch_id=config.cache_branch_id)

        with helper.enabled():
            t0 = time.time()
            with torch.inference_mode():
                out = self.pipe(prompt, num_inference_steps=num_steps, output_type="pil")
            elapsed = time.time()-t0
            stats = scheduler.get_stats()
        return out.images[0], elapsed, stats

    # ── 单 prompt 对比 ──

    def compare_single_prompt(self, prompt, num_steps=50, seed=42,
                              cache_interval=3, adaptive_config=None,
                              save_dir="results", display=True):
        os.makedirs(save_dir, exist_ok=True)
        print(f'  Prompt: "{prompt}"')
        print(f"  Steps: {num_steps}, Seed: {seed}\n")

        print("  [1/3] DDIM Baseline ...")
        img_bl, t_bl = self.generate_ddim(prompt, num_steps, seed)
        print(f"        {t_bl:.2f}s")

        print(f"  [2/3] DeepCache Official (interval={cache_interval}) ...")
        img_dc, t_dc = self.generate_deepcache(prompt, num_steps, seed, cache_interval)
        print(f"        {t_dc:.2f}s  ({t_bl/max(t_dc,1e-6):.2f}x)")

        print("  [3/3] Adaptive DeepCache ...")
        img_ad, t_ad, stats = self.generate_adaptive(prompt, num_steps, seed, adaptive_config)
        print(f"        {t_ad:.2f}s  ({t_bl/max(t_ad,1e-6):.2f}x)")
        print(f"        Schedule: {stats}")

        print("\n  Metrics ...")
        fid_dc = self.fid_calc.calculate_fid([img_bl],[img_dc])
        fid_ad = self.fid_calc.calculate_fid([img_bl],[img_ad])
        cl_bl = self.clip_scorer.score(img_bl, prompt)
        cl_dc = self.clip_scorer.score(img_dc, prompt)
        cl_ad = self.clip_scorer.score(img_ad, prompt)

        print(f"  FID  DC={fid_dc:.2f}  ADC={fid_ad:.2f}")
        print(f"  CLIP BL={cl_bl:.4f}  DC={cl_dc:.4f}  ADC={cl_ad:.4f}")

        comp = create_comparison_image(
            img_bl, img_dc, img_ad, prompt,
            (t_bl, t_dc, t_ad), (0.0, fid_dc, fid_ad), (cl_bl, cl_dc, cl_ad))

        safe = "".join(c if c.isalnum() or c in "_ " else "" for c in prompt[:40]).strip().replace(" ","_")
        path = os.path.join(save_dir, f"cmp_{safe}_s{seed}.png")
        comp.save(path, quality=95)
        print(f"  ✅ Saved → {path}")

        if display:
            try:
                from IPython.display import display as ipd
                ipd(comp)
            except ImportError: pass

        return {
            'prompt': prompt, 'seed': seed,
            'comparison_image': comp, 'comparison_path': path,
            'images': {'baseline': img_bl, 'deepcache': img_dc, 'adaptive': img_ad},
            'times': {'baseline': t_bl, 'deepcache': t_dc, 'adaptive': t_ad},
            'speedups': {'deepcache': t_bl/max(t_dc,1e-6), 'adaptive': t_bl/max(t_ad,1e-6)},
            'fid': {'deepcache': fid_dc, 'adaptive': fid_ad},
            'clip': {'baseline': cl_bl, 'deepcache': cl_dc, 'adaptive': cl_ad},
            'adaptive_stats': stats,
        }

    # ── 批量对比 ──

    def run_benchmark(self, prompts, seeds=None, num_steps=50,
                      cache_interval=3, adaptive_config=None,
                      save_dir="benchmark_results", display=True):
        if seeds is None: seeds = [42, 123, 456]
        os.makedirs(save_dir, exist_ok=True)
        all_res = []
        total = len(prompts)*len(seeds); idx = 0

        for prompt in prompts:
            for seed in seeds:
                idx += 1
                print(f"\n{'='*60}\n  Run {idx}/{total}\n{'='*60}")
                r = self.compare_single_prompt(
                    prompt, num_steps, seed, cache_interval,
                    adaptive_config, save_dir, display)
                all_res.append(r)

        self._print_summary(all_res)
        return all_res

    def _print_summary(self, results):
        n = len(results)
        def s(v): return f"{np.mean(v):.2f}±{np.std(v):.2f}"

        print(f"\n{'='*75}")
        print(f"  BENCHMARK SUMMARY ({n} runs)")
        print(f"{'='*75}")
        print(f"  {'Method':<28} {'Time':<16} {'Speedup':<12} {'FID ↓':<14} {'CLIP ↑'}")
        print(f"  {'-'*80}")

        t_bl = [r['times']['baseline'] for r in results]
        c_bl = [r['clip']['baseline'] for r in results]
        print(f"  {'DDIM Baseline':<28} {s(t_bl):<16} {'1.00x':<12} {'—':<14} {s(c_bl)}")

        for m, name in [('deepcache','DeepCache (Official)'),('adaptive','Adaptive DC (Ours)')]:
            t = [r['times'][m] for r in results]
            sp = [r['speedups'][m] for r in results]
            f = [r['fid'][m] for r in results]
            c = [r['clip'][m] for r in results]
            print(f"  {name:<28} {s(t):<16} {s(sp)+'x':<12} {s(f):<14} {s(c)}")

        print(f"{'='*75}")
        print(f"  FID ↓ = lower is better    CLIP ↑ = higher is better")
        print(f"{'='*75}")


# ============================================================
# 7. 入口
# ============================================================

def main():
    print("="*60)
    print("  Adaptive DeepCache Experiment")
    print("  DDIM vs DeepCache(Official) vs Adaptive DeepCache")
    print("="*60 + "\n")

    adaptive_config = AdaptiveCacheConfig(
        use_piecewise=True, use_continuous=False,
        phase_boundaries=[35, 15], phase_intervals=[1, 3, 5],
        cache_branch_id=0, total_steps=50,
    )

    prompts = [
        "a beautiful sunset over mountains, photorealistic, 8k",
        "A long time ago, in a galaxy far away...",
        "an astronaut riding a horse on mars, cinematic lighting",
        "a cozy coffee shop interior, warm colors, detailed",
    ]

    exp = AdaptiveDeepCacheExperiment(
        model_id="runwayml/stable-diffusion-v1-5", device=DEVICE)

    exp.run_benchmark(
        prompts=prompts, seeds=[42, 123, 456],
        num_steps=50, cache_interval=3,
        adaptive_config=adaptive_config,
        save_dir="adaptive_deepcache_results",
        display=True,
    )

    print("\n\nDone! Check 'adaptive_deepcache_results/'")


if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.